# 04 - Document Processing: Structured Extraction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai-arch/notebooks/04-document-processing.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:

1. Extract text from PDF documents programmatically
2. Understand image OCR concepts for document digitization
3. Use OpenAI's JSON mode for reliable structured output
4. Build Pydantic models for schema-driven extraction
5. Create a batch document processing pipeline
6. Validate and post-process extracted outputs

---

**Architecture Pattern:** Document processing pipelines ingest unstructured documents (PDFs, images, scans) and extract structured data using LLMs. The LLM acts as an intelligent parser that understands document semantics -- far more flexible than regex or rule-based extraction.

## Setup

Install the required packages.

In [ ]:
!pip install openai pydantic --quiet

## Configuration

In [ ]:
import os
import json
import re
import time
from typing import Optional
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    print("WARNING: OPENAI_API_KEY not set. Mock responses will be used.")
    print("To use the real API, set your key: export OPENAI_API_KEY='sk-...'")
else:
    print("API key found. Using live OpenAI API.")

client = OpenAI(api_key=api_key if api_key else "sk-mock-key")
MODEL = "gpt-4o-mini"

---
## 1. PDF Text Extraction

In production, you would use libraries like **PyPDF2**, **pdfplumber**, or cloud OCR services (Google Document AI, AWS Textract) to extract text from PDFs. Here we simulate the extracted text from different document types.

In [ ]:
# Simulated PDF text extraction
# In production: from PyPDF2 import PdfReader; reader = PdfReader('invoice.pdf')

def mock_pdf_extract(pdf_path):
    """Simulate PDF text extraction (would use PyPDF2 in production)."""
    print(f"Extracting text from: {pdf_path}")
    print("(Using simulated extraction -- in production, use PyPDF2/pdfplumber)")
    # Return pre-defined text based on filename
    return SAMPLE_DOCUMENTS.get(pdf_path, "No content found.")


SAMPLE_DOCUMENTS = {
    "invoice_001.pdf": """INVOICE
Invoice Number: INV-2024-0847
Date: March 15, 2024
Due Date: April 14, 2024

Bill To:
Acme Corporation
123 Business Ave, Suite 400
San Francisco, CA 94105

From:
TechSupply Inc.
456 Vendor Lane
Austin, TX 78701

Items:
1. Cloud Hosting (Annual)    $12,000.00
2. SSL Certificate (3-year)  $299.00
3. Support Plan (Premium)    $4,800.00

Subtotal: $17,099.00
Tax (8.25%): $1,410.67
Total: $18,509.67

Payment Terms: Net 30
Bank: First National Bank
Account: 9876543210
Routing: 021000021""",

    "contract_001.pdf": """SERVICE AGREEMENT

This Service Agreement ("Agreement") is entered into as of January 1, 2024,
by and between DataFlow Systems LLC ("Provider") and GlobalRetail Inc. ("Client").

1. SERVICES: Provider shall deliver cloud-based data analytics services
   including real-time dashboard, ETL pipeline management, and monthly reporting.

2. TERM: This Agreement shall commence on January 1, 2024 and continue
   for a period of twenty-four (24) months, ending December 31, 2025.

3. COMPENSATION: Client shall pay Provider a monthly fee of $15,000 USD,
   due on the first business day of each month. Late payments incur 1.5%
   monthly interest.

4. TERMINATION: Either party may terminate with 90 days written notice.
   Early termination fee: 3 months of service fees.

5. CONFIDENTIALITY: Both parties agree to maintain confidentiality of
   proprietary information for 3 years after termination.

6. LIABILITY: Provider's total liability shall not exceed 12 months of fees paid.

Signed:
DataFlow Systems LLC - Jane Smith, CEO
GlobalRetail Inc. - Robert Chen, CTO""",

    "resume_001.pdf": """SARAH JOHNSON
Senior Machine Learning Engineer
sarah.johnson@email.com | (555) 123-4567 | San Francisco, CA
LinkedIn: linkedin.com/in/sarahjohnson | GitHub: github.com/sjohnson

EXPERIENCE

Senior ML Engineer -- TechCorp AI (2022-Present)
- Led development of real-time recommendation system serving 10M+ users
- Reduced model inference latency by 40% through TensorRT optimization
- Built MLOps pipeline with Kubeflow, reducing deployment time from days to hours

ML Engineer -- DataStart Inc. (2019-2022)
- Developed NLP models for sentiment analysis achieving 94% accuracy
- Implemented A/B testing framework for model comparison
- Mentored 3 junior engineers

EDUCATION
M.S. Computer Science -- Stanford University (2019)
B.S. Mathematics -- UC Berkeley (2017)

SKILLS
Python, PyTorch, TensorFlow, Kubernetes, AWS, GCP, SQL, Spark"""
}

# Demo extraction
for filename in SAMPLE_DOCUMENTS:
    text = mock_pdf_extract(filename)
    print(f"  Extracted {len(text)} chars, {len(text.splitlines())} lines\n")

---
## 2. Image OCR Concepts

For scanned documents and images, OCR (Optical Character Recognition) is needed before LLM processing. Here is an overview of the pipeline and tools available.

In [ ]:
# OCR pipeline overview -- conceptual, no actual image processing
ocr_pipeline = {
    "step_1_intake": {
        "description": "Accept input documents (PDF, TIFF, JPEG, PNG)",
        "tools": ["Pillow", "pdf2image"],
    },
    "step_2_preprocessing": {
        "description": "Clean images: deskew, denoise, binarize, resize",
        "tools": ["OpenCV", "Pillow"],
    },
    "step_3_ocr": {
        "description": "Extract text from images using OCR engine",
        "tools": ["Tesseract OCR", "Google Document AI", "AWS Textract", "Azure Form Recognizer"],
    },
    "step_4_postprocess": {
        "description": "Clean OCR output: fix common errors, normalize whitespace",
        "tools": ["regex", "spaCy", "custom rules"],
    },
    "step_5_llm_extract": {
        "description": "Send cleaned text to LLM for structured extraction",
        "tools": ["OpenAI GPT-4o-mini", "JSON mode", "Pydantic validation"],
    }
}

print("=== Document OCR Pipeline ===")
for step, info in ocr_pipeline.items():
    print(f"\n{step.replace('_', ' ').title()}")
    print(f"  {info['description']}")
    print(f"  Tools: {', '.join(info['tools'])}")

print("\n" + "=" * 50)
print("In this notebook, we focus on steps 4-5: post-processing")
print("OCR output and using LLMs for structured extraction.")

---
## 3. Structured Output with JSON Mode

OpenAI's JSON mode (`response_format={"type": "json_object"}`) guarantees valid JSON output. This is essential for programmatic downstream processing -- no more parsing brittle free-text responses.

In [ ]:
def extract_json(document_text, document_type, fields):
    """Extract specific fields as JSON using OpenAI JSON mode."""
    fields_str = ", ".join(f'"{f}"' for f in fields)
    prompt = (
        f"Extract the following fields from this {document_type}: {fields_str}.\n\n"
        f"Return a JSON object with these exact keys. Use null for any field not found.\n"
        f"For monetary values, return as numbers (no $ sign). For dates, use YYYY-MM-DD format.\n\n"
        f"Document:\n---\n{document_text}\n---"
    )

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You extract structured data from documents. Always respond with valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=500,
            response_format={"type": "json_object"}
        )
        result = json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"API call failed: {e}")
        result = {
            "invoice_number": "INV-2024-0847",
            "date": "2024-03-15",
            "due_date": "2024-04-14",
            "vendor_name": "TechSupply Inc.",
            "client_name": "Acme Corporation",
            "subtotal": 17099.00,
            "tax": 1410.67,
            "total": 18509.67,
            "payment_terms": "Net 30"
        }
        print("Using mock response for demonstration.")

    return result


# Extract invoice fields
invoice_fields = [
    "invoice_number", "date", "due_date", "vendor_name",
    "client_name", "subtotal", "tax", "total", "payment_terms"
]

invoice_text = SAMPLE_DOCUMENTS["invoice_001.pdf"]
result = extract_json(invoice_text, "invoice", invoice_fields)

print("=== Structured JSON Extraction ===")
print(json.dumps(result, indent=2))
print(f"\nOutput is a Python dict -- ready for database insertion or API response.")

---
## 4. Schema-Driven Extraction with Pydantic

For production systems, define **Pydantic models** for each document type. This gives you:
- Type validation (dates are dates, numbers are numbers)
- Required vs optional fields
- Custom validators (business rules)
- Auto-generated JSON schema to guide the LLM

In [ ]:
from pydantic import BaseModel, Field, field_validator
from datetime import date


class LineItem(BaseModel):
    description: str
    amount: float


class InvoiceExtraction(BaseModel):
    invoice_number: str
    date: date
    due_date: date
    vendor_name: str
    client_name: str
    line_items: list[LineItem]
    subtotal: float
    tax: float
    total: float
    payment_terms: str = "Net 30"

    @field_validator("total")
    @classmethod
    def total_must_match(cls, v, info):
        expected = info.data.get("subtotal", 0) + info.data.get("tax", 0)
        if abs(v - expected) > 0.02:
            raise ValueError(f"Total {v} does not equal subtotal + tax ({expected})")
        return v


class ContractExtraction(BaseModel):
    provider: str
    client: str
    start_date: date
    end_date: date
    term_months: int
    monthly_fee: float
    termination_notice_days: int
    early_termination_fee_months: int
    liability_cap_months: int
    confidentiality_years: int
    services: list[str]


class ResumeExtraction(BaseModel):
    name: str
    title: str
    email: str
    phone: Optional[str] = None
    location: Optional[str] = None
    years_experience: int
    skills: list[str]
    education: list[str]
    current_company: Optional[str] = None


# Show the generated JSON schema
print("=== InvoiceExtraction JSON Schema ===")
print(json.dumps(InvoiceExtraction.model_json_schema(), indent=2)[:800])
print("...")

In [ ]:
def extract_with_schema(document_text, schema_model):
    """Extract data using a Pydantic schema to guide the LLM."""
    schema = json.dumps(schema_model.model_json_schema(), indent=2)
    prompt = (
        f"Extract data from the following document.\n"
        f"Return a JSON object matching this exact schema:\n\n{schema}\n\n"
        f"Document:\n---\n{document_text}\n---"
    )

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a document extraction engine. Return only valid JSON matching the provided schema."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=800,
            response_format={"type": "json_object"}
        )
        data = json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"API call failed: {e}")
        if schema_model == ContractExtraction:
            data = {
                "provider": "DataFlow Systems LLC",
                "client": "GlobalRetail Inc.",
                "start_date": "2024-01-01",
                "end_date": "2025-12-31",
                "term_months": 24,
                "monthly_fee": 15000.0,
                "termination_notice_days": 90,
                "early_termination_fee_months": 3,
                "liability_cap_months": 12,
                "confidentiality_years": 3,
                "services": ["real-time dashboard", "ETL pipeline management", "monthly reporting"]
            }
        elif schema_model == ResumeExtraction:
            data = {
                "name": "Sarah Johnson",
                "title": "Senior Machine Learning Engineer",
                "email": "sarah.johnson@email.com",
                "phone": "(555) 123-4567",
                "location": "San Francisco, CA",
                "years_experience": 5,
                "skills": ["Python", "PyTorch", "TensorFlow", "Kubernetes", "AWS", "GCP", "SQL", "Spark"],
                "education": ["M.S. Computer Science, Stanford University (2019)", "B.S. Mathematics, UC Berkeley (2017)"],
                "current_company": "TechCorp AI"
            }
        else:
            data = {}
        print("Using mock response for demonstration.")

    return schema_model.model_validate(data)


# Extract contract
contract = extract_with_schema(SAMPLE_DOCUMENTS["contract_001.pdf"], ContractExtraction)
print("=== Contract Extraction ===")
print(json.dumps(contract.model_dump(mode="json"), indent=2))
print(f"\nTotal contract value: ${contract.monthly_fee * contract.term_months:,.2f}")
print(f"Early exit cost: ${contract.monthly_fee * contract.early_termination_fee_months:,.2f}")

In [ ]:
# Extract resume
resume = extract_with_schema(SAMPLE_DOCUMENTS["resume_001.pdf"], ResumeExtraction)
print("=== Resume Extraction ===")
print(json.dumps(resume.model_dump(mode="json"), indent=2))
print(f"\nCandidate: {resume.name}")
print(f"Skills: {', '.join(resume.skills[:5])}...")
print(f"Experience: {resume.years_experience} years")

---
## 5. Batch Processing Pipeline

Process multiple documents through the extraction pipeline with logging, error handling, timing, and result aggregation.

In [ ]:
from dataclasses import dataclass, field as dc_field


@dataclass
class ProcessingResult:
    doc_id: str
    doc_type: str
    status: str  # "success", "error", "validation_failed"
    data: dict = dc_field(default_factory=dict)
    error: str = ""
    processing_time_ms: float = 0.0


# Schema registry -- maps document types to Pydantic models
SCHEMA_REGISTRY = {
    "contract": ContractExtraction,
    "resume": ResumeExtraction,
}


def process_document(doc_id, doc_type, text):
    """Process a single document through the extraction pipeline."""
    start = time.time()

    if doc_type not in SCHEMA_REGISTRY:
        return ProcessingResult(
            doc_id=doc_id, doc_type=doc_type,
            status="error", error=f"Unknown doc type: {doc_type}"
        )

    try:
        schema = SCHEMA_REGISTRY[doc_type]
        result = extract_with_schema(text, schema)
        elapsed = (time.time() - start) * 1000
        return ProcessingResult(
            doc_id=doc_id, doc_type=doc_type,
            status="success",
            data=result.model_dump(mode="json"),
            processing_time_ms=elapsed
        )
    except Exception as e:
        elapsed = (time.time() - start) * 1000
        return ProcessingResult(
            doc_id=doc_id, doc_type=doc_type,
            status="error", error=str(e),
            processing_time_ms=elapsed
        )


# Batch processing
batch = [
    ("DOC-001", "contract", SAMPLE_DOCUMENTS["contract_001.pdf"]),
    ("DOC-002", "resume", SAMPLE_DOCUMENTS["resume_001.pdf"]),
    ("DOC-003", "contract", SAMPLE_DOCUMENTS["contract_001.pdf"]),
    ("DOC-004", "resume", SAMPLE_DOCUMENTS["resume_001.pdf"]),
]

results = []
print(f"Processing {len(batch)} documents...\n")
print(f"{'ID':<10} {'Type':<12} {'Status':<18} {'Time (ms)':>10}")
print("-" * 52)

for doc_id, doc_type, text in batch:
    r = process_document(doc_id, doc_type, text)
    results.append(r)
    print(f"{r.doc_id:<10} {r.doc_type:<12} {r.status:<18} {r.processing_time_ms:>8.1f}ms")

success = sum(1 for r in results if r.status == "success")
failed = len(results) - success
avg_time = sum(r.processing_time_ms for r in results) / len(results)
print(f"\n--- Summary ---")
print(f"Total: {len(results)} | Success: {success} | Failed: {failed}")
print(f"Avg processing time: {avg_time:.1f}ms")

---
## 6. Output Validation

After LLM extraction, validate the output with business rules before storing. Pydantic handles type validation, but you also need domain-specific checks.

In [ ]:
def validate_contract(data):
    """Run business-rule validations on an extracted contract."""
    issues = []

    # Date consistency
    if data.get("start_date") and data.get("end_date"):
        start = str(data["start_date"])
        end = str(data["end_date"])
        if start >= end:
            issues.append("Start date must be before end date.")

    # Fee validation
    if data.get("monthly_fee", 0) <= 0:
        issues.append("Monthly fee must be positive.")

    # Required fields
    for field_name in ["provider", "client", "services"]:
        if not data.get(field_name):
            issues.append(f"Missing required field: {field_name}")

    # Services should not be empty
    if isinstance(data.get("services"), list) and len(data["services"]) == 0:
        issues.append("Services list is empty.")

    return issues


def validate_resume(data):
    """Run validations on an extracted resume."""
    issues = []

    # Email format
    if data.get("email") and not re.match(r"[^@]+@[^@]+\.[^@]+", data["email"]):
        issues.append(f"Invalid email format: {data['email']}")

    # Skills not empty
    if not data.get("skills"):
        issues.append("No skills extracted.")

    # Reasonable experience
    if data.get("years_experience", 0) > 50:
        issues.append(f"Unreasonable years of experience: {data['years_experience']}")

    return issues


VALIDATORS = {
    "contract": validate_contract,
    "resume": validate_resume,
}

# Validate all successful results
print("=== Post-Extraction Validation ===")

for r in results:
    if r.status != "success":
        continue

    validator = VALIDATORS.get(r.doc_type)
    if validator:
        issues = validator(r.data)
        status = "PASS" if not issues else "WARN"
        print(f"\n{r.doc_id} ({r.doc_type}): {status}")
        if issues:
            for issue in issues:
                print(f"  WARNING: {issue}")
        else:
            print(f"  All validations passed.")

In [ ]:
# Confidence scoring -- ask the LLM to audit its own extraction
def score_confidence(document_text, extracted):
    """Ask the LLM to score confidence on each extracted field."""
    prompt = (
        f"You are a document extraction auditor. Compare the extracted data against "
        f"the original document. For each field, assign a confidence score (0.0 to 1.0).\n\n"
        f'Return JSON: {{"field_name": {{"confidence": 0.95, "issue": null}}, ...}}\n\n'
        f"Extracted data:\n{json.dumps(extracted, indent=2, default=str)}\n\n"
        f"Original document:\n---\n{document_text[:1500]}\n---"
    )

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=500,
            response_format={"type": "json_object"}
        )
        result = json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"API call failed: {e}")
        result = {
            field_name: {"confidence": 0.95 if i < 5 else 0.85, "issue": None}
            for i, field_name in enumerate(extracted.keys())
        }
        print("Using mock response for demonstration.")

    return result


# Score the contract extraction
scores = score_confidence(
    SAMPLE_DOCUMENTS["contract_001.pdf"],
    contract.model_dump(mode="json")
)

print("=== Extraction Confidence Scores ===")
print(f"{'Field':<30} {'Confidence':>10} {'Status':>8}")
print("-" * 50)

needs_review = []
for field_name, info in scores.items():
    conf = info.get("confidence", 0)
    status = "OK" if conf >= 0.9 else "REVIEW" if conf >= 0.7 else "FAIL"
    print(f"{field_name:<30} {conf:>8.0%} {status:>8}")
    if conf < 0.9:
        needs_review.append(field_name)

if needs_review:
    print(f"\nFields requiring human review: {', '.join(needs_review)}")
else:
    print(f"\nAll fields above confidence threshold -- auto-approved.")

---
## Summary

This notebook built a complete document processing pipeline:

| Stage | What It Does | Key Tools |
|---|---|---|
| PDF Extraction | Get text from documents | PyPDF2, pdfplumber |
| OCR | Digitize scanned documents | Tesseract, Document AI |
| JSON Mode | Guaranteed valid JSON output | `response_format={"type": "json_object"}` |
| Schema-Driven | Pydantic models guide and validate extraction | Pydantic, JSON Schema |
| Batch Processing | Pipeline with error handling and timing | Custom pipeline class |
| Validation | Business rules + confidence scoring | Pydantic validators, LLM audit |

**Key takeaways:**
- JSON mode eliminates parsing failures from free-text LLM output
- Pydantic schemas guide the LLM AND validate the output -- double duty
- Always validate LLM extractions with business rules, not just type checks
- Confidence scoring enables human-in-the-loop for uncertain extractions

**Next:** In notebook 05, we build a multi-model router that routes requests to different models based on complexity and cost.